In [2]:
import sys
sys.path.append(r'C:\Users\Nikhil Kapoor\Documents\dataScienceProjects\src')
from tennis_stats import win_rate_on, head_to_head

from importlib import reload
import tennis_stats; reload(tennis_stats)
from tennis_stats import win_rate_on, head_to_head

import json, os
os.makedirs('data', exist_ok=True)


In [3]:
import pandas as pd
URL = 'https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_2010.csv'
df = pd.read_csv(URL)

In [4]:
import sys
sys.path.append(r'C:\Users\Nikhil Kapoor\Documents\dataScienceProjects\src')
from tennis_stats import win_rate_on, head_to_head

In [5]:
unique_players = set(df['winner_name']).union(set(df['loser_name']))
print(df.shape, len(unique_players), 'unique_players')



(3030, 49) 472 unique_players


In [6]:
top10 = (df.groupby('winner_name')
         .size()
         .sort_values(ascending=False)
         .head(10)
         .reset_index())
top10.columns = ['winner_name','wins']

In [7]:
wins = (df.groupby('winner_name')
       .size()
       .rename('wins'))
losses = (df.groupby('loser_name')
       .size()
       .rename('losses'))
aces = (df.groupby('winner_name')['w_ace']
       .sum()
       .rename('aces'))
avg_min = (df.groupby('winner_name')['minutes']
       .mean()
       .rename('avg_min_won'))


In [8]:
stats = pd.concat([wins,losses,aces,avg_min], axis = 1).fillna(0)
stats['win_rate'] = stats['wins']/ (stats['wins'] + stats['losses'])
stats = stats.sort_values('wins', ascending=False).head(10)

records = stats.reset_index().to_dict(orient='records')

with open('data/top10_stats_2010.json', 'w') as f:
    json.dump(records, f, indent=2, default=str)

In [9]:
with open('data/top10_stats_2010.json') as f:
            loaded = json.load(f)

top_ace = max(loaded, key=lambda r: r['aces'])
print(f"{top_ace['index']}: {int(top_ace['aces'])} aces")


Andy Roddick: 610 aces


In [10]:
top10_names = stats.index.tolist()
pivot = pd.DataFrame({
    'Hard' : [win_rate_on(df, p, 'Hard') for p in top10_names],
    'Clay' : [win_rate_on(df, p, 'Clay') for p in top10_names],
    'Grass' : [win_rate_on(df, p, 'Grass') for p in top10_names],
}, index=top10_names
)

for name in top10_names:
    overall = win_rate_on(df, name)
    clay = win_rate_on(df, name, 'Clay')
    print(f"{name:25s} overall= {overall:.2f} clay={clay:.2f}")


Rafael Nadal              overall= 0.88 clay=1.00
Roger Federer             overall= 0.84 clay=0.71
Novak Djokovic            overall= 0.78 clay=0.75
David Ferrer              overall= 0.71 clay=0.82
Robin Soderling           overall= 0.72 clay=0.70
Jurgen Melzer             overall= 0.68 clay=0.68
Andy Roddick              overall= 0.73 clay=0.67
Gael Monfils              overall= 0.70 clay=0.67
Andy Murray               overall= 0.72 clay=0.60
Tomas Berdych             overall= 0.63 clay=0.78


In [11]:
import os
os.makedirs('data', exist_ok=True)
stats.to_csv('data/top10_stats_2010.csv')
# Add data/ to .gitignore if it isn't already — raw CSVs shouldn't live in git.

In [12]:
spread = pivot.max(axis=1) - pivot.min(axis=1)
spread.sort_values(ascending=False).head(3)


Tomas Berdych    0.313665
Gael Monfils     0.220000
David Ferrer     0.196742
dtype: float64

29-05-2026

In [14]:
fed_all = df[(df['winner_name'] == 'Roger Federer') | (df['loser_name'] == 'Roger Federer')]
fed_longest = fed_all.sort_values('minutes', ascending=False).head(1)
fed_longest[['tourney_name','round','winner_name','loser_name','surface','minutes']]

,tourney_name,round,winner_name,loser_name,surface,minutes
2462,US Open,SF,Novak Djokovic,Roger Federer,Hard,224.0


In [ ]:

print(head_to_head(df, 'Rafael Nadal', 'Roger Federer'))
print(head_to_head(df, 'Rafael Nadal', 'Novak Djokovic'))


In [ ]:
try:
    win_rate_on(df, 'Nadal','Clay')
except ValueError as e:
    print(f"Caught: {e}")

try:
    head_to_head(df, 'Rafael Nadal', 'Jannik Sinner')
except ValueError as e:
    print(f"Caught: {e}")

In [ ]:
big_four = ['Rafael Nadal', 'Roger Federer', 'Andy Murray', 'Noval Djokovic']

pd.pivot_table(
    df[df['winner_name'].isin(big_four)],
    values='minutes',
    index='winner_name',
    columns='surface',
    aggfunc='mean'
)


In [ ]:
reload(tennis_stats)
from tennis_stats import top_players_by_wins

for name, wins in top_players_by_wins(df):
    print(f"{name:25s} {wins}")